로지스틱 회귀의 기본 컨셉과 선형회귀와의 차이점을 기반으로 Plotly를 이용하여 시각화를 통해 설명하세요

In [1]:
import numpy as np
import plotly.graph_objects as go

# 데이터 생성
x = np.linspace(-10, 10, 100)

# 선형회귀 (직선)
y_linear = 0.5 * x

# 로지스틱 함수 (sigmoid)
y_logistic = 1 / (1 + np.exp(-0.5 * x))

# 그래프 생성
fig = go.Figure()

# 선형회귀
fig.add_trace(go.Scatter(
    x=x,
    y=y_linear,
    mode='lines',
    name='Linear Regression'
))

# 로지스틱 회귀
fig.add_trace(go.Scatter(
    x=x,
    y=y_logistic,
    mode='lines',
    name='Logistic Regression'
))

# 레이아웃
fig.update_layout(
    title="Linear vs Logistic Regression",
    xaxis_title="X",
    yaxis_title="Output",
)

fig.show()

진짜로 돌아가는” NumPy만으로 만든 최소 신경망 + 오차역전파
(입력→은닉→출력 2층 네트워크, 이진분류 예제 / 주석은 최대한 쉽게)
✔️ Forward (앞으로 계산)
입력 → 예측값 만들기
✔️ Loss
“얼마나 틀렸는지”
✔️ Backprop
“어디를 얼마나 고칠지 계산”
✔️ Update
실제로 가중치 수정
5️⃣ 실무 관점 해석 : 이 코드가 하는 일
👉 “데이터 경계를 자동으로 학습”
처음: 랜덤 → 엉망
반복: 점점 정확
결과: 분류 가능
6️⃣ 한 줄 핵심 정리
👉 “Backprop은 오차를 이용해 가중치를 조금씩 수정해서 정답에 가까워지게 만드는 과정”
👉 “틀린 만큼, 책임 있는 가중치만 조금씩 벌준다”

In [ ]:
#(forward + backprop 직접 구현)
import numpy as np

# ============================================
# [0] 재현성 (항상 같은 결과)
# ============================================
np.random.seed(42)

# ============================================
# [1] 데이터 만들기 (장난감 데이터)
# ============================================
# 입력 X: 2차원 (예: x1, x2)
# 정답 y: 0 또는 1 (이진 분류)
# 간단히 "선으로 나뉘는 데이터" 생성
N = 200
X = np.random.randn(N, 2)
y = (X[:, 0] + X[:, 1] > 0).astype(int).reshape(-1, 1)  # 경계: x1+x2=0

# ============================================
# [2] 하이퍼파라미터
# ============================================
input_dim = 2
hidden_dim = 8
output_dim = 1
lr = 0.1            # 학습률 (얼마나 크게 수정할지)
epochs = 2000       # 반복 횟수

# ============================================
# [3] 가중치 초기화
# ============================================
# 작은 난수로 시작 (너무 크면 학습 불안정)
W1 = 0.1 * np.random.randn(input_dim, hidden_dim)
b1 = np.zeros((1, hidden_dim))

W2 = 0.1 * np.random.randn(hidden_dim, output_dim)
b2 = np.zeros((1, output_dim))

# ============================================
# [4] 활성화 함수 정의
# ============================================
def sigmoid(x):
    # 0~1 사이로 눌러주는 함수 (확률처럼 해석 가능) / [W1, b1] → Z1 → sigmoid → A1
    return 1 / (1 + np.exp(-x))

def sigmoid_grad(s):
    # sigmoid의 미분값
    # s = sigmoid(x)일 때 s*(1-s) / [W2, b2] → Z2 → sigmoid → A2 (예측)
    return s * (1 - s)

# ============================================
# [5] 학습 루프 (Forward → Loss → Backprop → Update) / Loss 계산
# ============================================
for epoch in range(epochs):

    # -----------------------------
    # (A) Forward Propagation
    # -----------------------------
    # 1층
    Z1 = X @ W1 + b1          # 선형변환
    A1 = sigmoid(Z1)          # 비선형 (은닉층 활성화)

    # 2층 (출력층)
    Z2 = A1 @ W2 + b2
    A2 = sigmoid(Z2)          # 최종 출력 (0~1 확률)

    # -----------------------------
    # (B) Loss 계산 (Binary Cross Entropy)
    # -----------------------------
    # y: 정답, A2: 예측
    eps = 1e-8  # log(0) 방지
    loss = -np.mean(y * np.log(A2 + eps) + (1 - y) * np.log(1 - A2 + eps))

    # -----------------------------
    # (C) Backpropagation  : 오차 계산 (A2 - y)
    # -----------------------------
    # 출력층 gradient
    # dL/dZ2
    dZ2 = A2 - y  # (BCE + sigmoid 조합에서 깔끔하게 나옴)

    # dL/dW2, dL/db2
    dW2 = (A1.T @ dZ2) / N
    db2 = np.sum(dZ2, axis=0, keepdims=True) / N

    # 은닉층으로 역전파
    dA1 = dZ2 @ W2.T
    dZ1 = dA1 * sigmoid_grad(A1)  # 체인룰

    # dL/dW1, dL/db1
    dW1 = (X.T @ dZ1) / N
    db1 = np.sum(dZ1, axis=0, keepdims=True) / N

    # -----------------------------
    # (D) 파라미터 업데이트
    # -----------------------------
    W1 -= lr * dW1
    b1 -= lr * db1
    W2 -= lr * dW2
    b2 -= lr * db2

    # -----------------------------
    # (E) 중간 로그
    # -----------------------------
    if epoch % 200 == 0:
        print(f"epoch {epoch:4d} | loss {loss:.4f}")

# ============================================
# [6] 학습 결과 확인
# ============================================
pred = (A2 > 0.5).astype(int)
acc = (pred == y).mean()
print("\n최종 정확도:", acc)

epoch    0 | loss 0.7124
epoch  200 | loss 0.6107
epoch  400 | loss 0.3218
epoch  600 | loss 0.1850
epoch  800 | loss 0.1304
epoch 1000 | loss 0.1020
epoch 1200 | loss 0.0847
epoch 1400 | loss 0.0729
epoch 1600 | loss 0.0644
epoch 1800 | loss 0.0580

최종 정확도: 1.0
